# Project 3 Evidence
Run top-to-bottom and capture screenshots only on cells marked **[SCREENSHOT]**.

## 0) Setup

In [10]:
import os
import sys
sys.path.insert(0, "/home/jovyan/project/work")
from lakehouse_spark import new_session
spark = new_session("project3_evidence")
spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

Spark ready


## 1) CDC correctness 
Counts must match: PostgreSQL vs Iceberg silver.

In [11]:
import psycopg2

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname=os.environ.get("PG_DB", "sourcedb"),
    user=os.environ.get("PG_USER", "cdc_user"),
    password=os.environ.get("PG_PASSWORD", "admin"),
)
cur = conn.cursor()
cur.execute("SELECT count(*) FROM customers")
pg_customers = cur.fetchone()[0]
cur.execute("SELECT count(*) FROM drivers")
pg_drivers = cur.fetchone()[0]
cur.close(); conn.close()

sf_customers = spark.sql("SELECT count(*) AS c FROM lakehouse.cdc.silver_customers").collect()[0]["c"]
sf_drivers = spark.sql("SELECT count(*) AS c FROM lakehouse.cdc.silver_drivers").collect()[0]["c"]

print("postgres.customers:", pg_customers, "| silver_customers:", sf_customers)
print("postgres.drivers  :", pg_drivers, "| silver_drivers  :", sf_drivers)

postgres.customers: 12 | silver_customers: 12
postgres.drivers  : 8 | silver_drivers  : 0


## 2) CDC spot-check rows 

In [12]:
spark.sql("SELECT id,name,email,country FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5").show(truncate=False)
spark.sql("SELECT id,name,license_number,rating,city,active FROM lakehouse.cdc.silver_drivers ORDER BY id LIMIT 5").show(truncate=False)

+---+--------------+-------------------------+---------+
|id |name          |email                    |country  |
+---+--------------+-------------------------+---------+
|1  |Alice Mets    |alice.updated@example.com|Estonia  |
|2  |Bob Virtanen  |bob@example.com          |Finland  |
|4  |David Jonaitis|david@example.com        |Lithuania|
|5  |Eva Svensson  |eva@example.com          |Sweden   |
|6  |Frank Muller  |frank@example.com        |Germany  |
+---+--------------+-------------------------+---------+

+---+----+--------------+------+----+------+
|id |name|license_number|rating|city|active|
+---+----+--------------+------+----+------+
+---+----+--------------+------+----+------+



## 3) Taxi silver schema + sample 

In [13]:
spark.sql("DESCRIBE lakehouse.taxi.silver_trips").show(truncate=False)
spark.sql("""
SELECT PULocationID, pickup_zone, dropoff_zone, trip_duration_minutes, trip_speed_kmh
FROM lakehouse.taxi.silver_trips
LIMIT 10
""").show(truncate=False)

+---------------------+---------+-------+
|col_name             |data_type|comment|
+---------------------+---------+-------+
|trip_id              |bigint   |NULL   |
|PULocationID         |int      |NULL   |
|DOLocationID         |int      |NULL   |
|fare_amount          |double   |NULL   |
|tip_amount           |double   |NULL   |
|payment_type         |int      |NULL   |
|trip_distance        |double   |NULL   |
|tpep_pickup_datetime |timestamp|NULL   |
|tpep_dropoff_datetime|timestamp|NULL   |
|trip_duration_minutes|double   |NULL   |
|trip_speed_kmh       |double   |NULL   |
|pickup_zone          |string   |NULL   |
|dropoff_zone         |string   |NULL   |
+---------------------+---------+-------+

+------------+-----------------------------+------------------------+---------------------+------------------+
|PULocationID|pickup_zone                  |dropoff_zone            |trip_duration_minutes|trip_speed_kmh    |
+------------+-----------------------------+-------------------

## 4) Core table counts 

In [14]:
spark.sql("SELECT count(*) AS cdc_customers_cnt FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT count(*) AS cdc_drivers_cnt FROM lakehouse.cdc.silver_drivers").show()
spark.sql("SELECT count(*) AS taxi_silver_cnt FROM lakehouse.taxi.silver_trips").show()
spark.sql("SELECT count(*) AS gold_tipping_behavior_cnt FROM lakehouse.taxi.gold_tipping_behavior").show()
spark.sql("SELECT count(*) AS gold_tip_rankings_cnt FROM lakehouse.taxi.gold_tip_rankings").show()
spark.sql("SELECT count(*) AS gold_driver_tip_report_cnt FROM lakehouse.taxi.gold_driver_tip_report").show()

+-----------------+
|cdc_customers_cnt|
+-----------------+
|               12|
+-----------------+

+---------------+
|cdc_drivers_cnt|
+---------------+
|              0|
+---------------+

+---------------+
|taxi_silver_cnt|
+---------------+
|        6564461|
+---------------+

+-------------------------+
|gold_tipping_behavior_cnt|
+-------------------------+
|                     5816|
+-------------------------+

+---------------------+
|gold_tip_rankings_cnt|
+---------------------+
|                 1198|
+---------------------+

+--------------------------+
|gold_driver_tip_report_cnt|
+--------------------------+
|                         0|
+--------------------------+



## 5) Tipping behavior output

In [15]:
spark.sql("""
SELECT *
FROM lakehouse.taxi.gold_tipping_behavior
ORDER BY avg_tip_percentage DESC
LIMIT 20
""").show(truncate=False)

+------------+-----------+------------------+------------------+------------------+------------------+----------------------------+-----------+
|PULocationID|pickup_hour|avg_tip_amount    |avg_tip_percentage|pct_zero_tip      |pct_high_tip      |high_tip_common_payment_type|trips_count|
+------------+-----------+------------------+------------------+------------------+------------------+----------------------------+-----------+
|265         |15         |5.380641025641024 |5135.390011008079 |56.41025641025641 |24.358974358974358|1                           |78         |
|1           |20         |49.666666666666664|1408.1560283687943|33.33333333333333 |66.66666666666666 |1                           |3          |
|8           |22         |32.5              |127.95275590551182|50.0              |50.0              |1                           |2          |
|246         |14         |3.0254578328813024|123.47657351286253|23.29846312798657 |62.50807180679323 |1                           |7743 

## 6) Best zone + hour query

In [16]:
spark.sql("""
SELECT PULocationID, pickup_hour, avg_tip_percentage
FROM lakehouse.taxi.gold_tipping_behavior
ORDER BY avg_tip_percentage DESC
LIMIT 1
""").show(truncate=False)

+------------+-----------+------------------+
|PULocationID|pickup_hour|avg_tip_percentage|
+------------+-----------+------------------+
|265         |15         |5135.390011008079 |
+------------+-----------+------------------+



## 7) Driver rating vs tip correlation

In [17]:
spark.sql("""
SELECT driver_rating, AVG(avg_tip_pct) AS mean_tip_pct, COUNT(*) AS drivers
FROM lakehouse.taxi.gold_driver_tip_report
GROUP BY driver_rating
ORDER BY driver_rating DESC
""").show(truncate=False)

+-------------+------------+-------+
|driver_rating|mean_tip_pct|drivers|
+-------------+------------+-------+
+-------------+------------+-------+



## 8) Snapshot history

In [18]:
spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM lakehouse.cdc.silver_customers.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|5502655263978847187|2026-05-03 14:43:30.927|append   |
|3463647024213026548|2026-05-03 13:54:28.747|append   |
|4164585981968936103|2026-05-03 13:50:54.385|append   |
|402294560137066004 |2026-05-03 13:46:01.906|append   |
|70591757547223788  |2026-05-03 13:36:21.196|append   |
+-------------------+-----------------------+---------+



## 9) Time travel proof 
Replace `snap_id` with one snapshot from cell above.

In [19]:
snap_id = None
if snap_id is None:
    print("Set snap_id to a real snapshot_id from previous cell and run again.")
else:
    spark.sql(f"SELECT count(*) AS rows_before FROM lakehouse.cdc.silver_customers VERSION AS OF {snap_id}").show()

Set snap_id to a real snapshot_id from previous cell and run again.


## Airflow UI screenshots still required
- **[SCREENSHOT 10]** Graph view with all tasks
- **[SCREENSHOT 11]** DAG runs list with >=3 consecutive success
- **[SCREENSHOT 12]** failed run + recovery run

## 10) Fix missing drivers + rebuild gold **[RUN THIS]**

Use these cells if `silver_drivers` or `gold_driver_tip_report` count is 0.

In [46]:
import subprocess, sys

subprocess.run([sys.executable, "/home/jovyan/project/work/bronze.py"], check=True)
subprocess.run([sys.executable, "/home/jovyan/project/work/silver.py"], check=True)
print("Rebuilt CDC bronze/silver (customers + drivers)")

Rebuilt CDC bronze/silver (customers + drivers)


In [47]:
subprocess.run([sys.executable, "/home/jovyan/project/work/taxi_gold.py"], check=True)
print("Rebuilt gold tables")

Rebuilt gold tables


## 11) Driver-related proof after rebuild 

In [48]:
import os
import sys
import subprocess
import psycopg2


def counts():
    sf = spark.sql("SELECT count(*) AS c FROM lakehouse.cdc.silver_drivers").collect()[0]["c"]
    gd = spark.sql("SELECT count(*) AS c FROM lakehouse.taxi.gold_driver_tip_report").collect()[0]["c"]
    return sf, gd


conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname=os.environ.get("PG_DB", "sourcedb"),
    user=os.environ.get("PG_USER", "cdc_user"),
    password=os.environ.get("PG_PASSWORD", "admin"),
)
conn.autocommit = True
cur = conn.cursor()
cur.execute("SELECT count(*) FROM drivers")
pg_drivers = cur.fetchone()[0]

sf_drivers, gold_driver = counts()
print("Before rebuild -> postgres.drivers:", pg_drivers, "| silver_drivers:", sf_drivers, "| gold_driver_tip_report:", gold_driver)

if sf_drivers == 0:
    cur.execute("UPDATE drivers SET rating = rating + 0.01 WHERE id IN (1,2,3)")
    cur.execute(
        "INSERT INTO drivers (name, license_number, rating, city, active) VALUES (%s,%s,%s,%s,%s)",
        ("Auto Driver", "TLC-29999", 4.71, "Manhattan", True),
    )
    print("Generated driver CDC events in Postgres.")

cur.close()
conn.close()

subprocess.run([sys.executable, "/home/jovyan/project/work/bronze.py"], check=True)
subprocess.run([sys.executable, "/home/jovyan/project/work/silver.py"], check=True)
subprocess.run([sys.executable, "/home/jovyan/project/work/taxi_gold.py"], check=True)

sf_drivers, gold_driver = counts()
print("After rebuild  -> postgres.drivers:", pg_drivers, "| silver_drivers:", sf_drivers)
print("After rebuild  -> gold_driver_tip_report:", gold_driver)

spark.sql("SELECT id,name,license_number,rating,city,active FROM lakehouse.cdc.silver_drivers ORDER BY id LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM lakehouse.taxi.gold_driver_tip_report ORDER BY driver_rating DESC, avg_tip_pct DESC LIMIT 20").show(truncate=False)

Before rebuild -> postgres.drivers: 8 | silver_drivers: 0 | gold_driver_tip_report: 0
Generated driver CDC events in Postgres.
After rebuild  -> postgres.drivers: 8 | silver_drivers: 4
After rebuild  -> gold_driver_tip_report: 3
+---+------------+--------------+------+---------+------+
|id |name        |license_number|rating|city     |active|
+---+------------+--------------+------+---------+------+
|1  |Tom Driver  |TLC-10001     |NULL  |Manhattan|true  |
|2  |Sarah Wheels|TLC-10002     |NULL  |Brooklyn |true  |
|3  |Mike Road   |TLC-10003     |NULL  |Queens   |true  |
|9  |Auto Driver |TLC-29999     |NULL  |Manhattan|true  |
+---+------------+--------------+------+---------+------+

+---------+-------------+------------------+------------------+-----------+
|driver_id|driver_rating|avg_tip_amount    |avg_tip_pct       |trips_count|
+---------+-------------+------------------+------------------+-----------+
|3        |NULL         |3.016711265528794 |19.66024611863266 |1641148    |
|1

## 12) Time travel final proof 

In [30]:
snaps = spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM lakehouse.cdc.silver_customers.snapshots
ORDER BY committed_at DESC
""")
snaps.show(truncate=False)

rows = snaps.collect()
if len(rows) < 2:
    print("Need at least 2 snapshots for time-travel proof. Trigger DAG once more and rerun.")
else:
    snap_id = rows[1]["snapshot_id"]
    print("Using previous snapshot_id:", snap_id)
    spark.sql(f"SELECT count(*) AS rows_before FROM lakehouse.cdc.silver_customers VERSION AS OF {snap_id}").show()

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|8096767299429510495|2026-05-03 15:09:44.604|append   |
|5470092522160328546|2026-05-03 15:08:23.505|append   |
|7623495354151015421|2026-05-03 15:06:43.446|append   |
|4239457405281786402|2026-05-03 15:03:01.805|append   |
|5502655263978847187|2026-05-03 14:43:30.927|append   |
|3463647024213026548|2026-05-03 13:54:28.747|append   |
|4164585981968936103|2026-05-03 13:50:54.385|append   |
|402294560137066004 |2026-05-03 13:46:01.906|append   |
|70591757547223788  |2026-05-03 13:36:21.196|append   |
+-------------------+-----------------------+---------+

Using previous snapshot_id: 5470092522160328546
+-----------+
|rows_before|
+-----------+
|         12|
+-----------+

